# 🚀 Context-Aware Prompting Experiment (261 Samples)
This experiment generates 261 Gherkin files using **project-branched prompts (Domain-Specific Personas)**.

**Input:** `full_user_stories_261.csv` and `extended_ground_truth.csv`
**Output:** `results/full_gpt54mini_output.csv`

In [ ]:
!pip install openai pandas tqdm

## 1. Upload data

In [ ]:
from google.colab import files
import pandas as pd
import os

print("Please upload 2 files: full_user_stories.csv and full_ground_truth.csv")
uploaded = files.upload()

df_us = pd.read_csv('full_user_stories.csv')
df_gt = pd.read_csv('full_ground_truth.csv')
print(f"Loaded {len(df_us)} samples.")

## 2. Dynamic Few-shot samples from the Ground Truth

In [ ]:
# Extract 3 representative samples for the 3 domains
def get_sample(id):
    us = df_us[df_us['id'] == id].iloc[0]['user_story']
    gk = df_gt[df_gt['id'] == id].iloc[0]['gherkin_content'].replace('\\n', '\n')
    return us, gk

us_eco, gk_eco = get_sample(2)   # Sylius (E-commerce)
us_fin, gk_fin = get_sample(102)  # Fineract (Finance)
us_soc, gk_soc = get_sample(202)  # Diaspora (Social Network)

## 3. Initialize the API and the Context-Aware Prompt Function

In [ ]:
from getpass import getpass
import openai

API_KEY = getpass('Enter OpenAI API Key: ')
client = openai.OpenAI(api_key=API_KEY)

def get_context_aware_prompt(domain, user_story):
    if domain == 'E-commerce':
        sys_msg = "You are an expert QA Engineer for Sylius, an E-commerce platform using the Behat framework. You write business-focused Gherkin tests. Focus on product variants, shopping carts, and UI interactions using business language."
        ex_us, ex_gk = us_eco, gk_eco
    elif domain == 'Finance/Banking':
        sys_msg = "You are an expert QA Engineer for Apache Fineract, a core banking system. Your signature style is using detailed Data Tables in the Background block to mock extensive financial data (e.g. LoanProduct, clients). Write highly technical, data-driven Gherkin."
        ex_us, ex_gk = us_fin, gk_fin
    else:
        sys_msg = "You are an expert QA Engineer for Diaspora, a Ruby on Rails social network using Capybara and Cucumber. Your signature style is writing concrete UI automation steps using CSS selectors (e.g. 'within \".class\"') and exact UI actions."
        ex_us, ex_gk = us_soc, gk_soc
        
    prompt = f"""Generate a BDD Gherkin file for the following user story. 
Follow the exact style, wording, and conventions shown in the Example.

EXAMPLE USER STORY:
{ex_us}

EXAMPLE GHERKIN:
{ex_gk}

---
NOW GENERATE FOR THIS USER STORY:
{user_story}
"""
    return sys_msg, prompt

## 4. Run the Experiment (Generate 261 Gherkin files)

In [ ]:
from tqdm import tqdm
import time

results = []
OUT_FILE = 'full_gpt54mini_output.csv'

for idx, row in tqdm(df_us.iterrows(), total=len(df_us)):
    sys_msg, user_prompt = get_context_aware_prompt(row['domain'], row['user_story'])
    
    success = False
    while not success:
        try:
            response = client.chat.completions.create(
                model="gpt-5.4-mini-2026-03-17",
                messages=[
                    {"role": "system", "content": sys_msg},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0
            )
            gen_text = response.choices[0].message.content.strip()
            
            # Strip markdown code blocks if the AI inserts them
            if gen_text.startswith("```gherkin"):
                gen_text = gen_text.replace("```gherkin", "", 1)
            if gen_text.startswith("```"):
                gen_text = gen_text.replace("```", "", 1)
            if gen_text.endswith("```"):
                gen_text = gen_text[:-3]
            gen_text = gen_text.strip()
            
            # SAVE API LOG
            import json
            from datetime import datetime, timedelta, timezone
            prompt_tokens = response.usage.prompt_tokens
            completion_tokens = response.usage.completion_tokens
            cost = (prompt_tokens / 1000000) * 0.15 + (completion_tokens / 1000000) * 0.60
            
            log_entry = {
                "id": row['id'],
                "timestamp": (datetime.now(timezone.utc) + timedelta(hours=7)).isoformat(),
                "model": "gpt-5.4-mini-2026-03-17",
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "total_tokens": prompt_tokens + completion_tokens,
                "cost_usd": cost
            }
            with open('full_api_log.jsonl', 'a', encoding='utf-8') as f:
                f.write(json.dumps(log_entry) + '\n')
            
            success = True
        except Exception as e:
            print(f"\nError at ID {row['id']} - Waiting 60s and retrying... {e}")
            time.sleep(60)
            
    results.append({
        'id': row['id'],
        'user_story': row['user_story'],
        'generated_gherkin': gen_text
    })
    time.sleep(1)
    
    # Auto-save
    pd.DataFrame(results).to_csv(OUT_FILE, index=False, encoding='utf-8')

print("\nCompleted 261 samples!")
files.download(OUT_FILE)